## **DX602 Final Project**

**Residential Energy Consumption Analysis**

Dataset: Option 2 — Environmental / Energy (home_data_final.csv)


# **Section 1: Introduction**
This project analyzes residential energy consumption data from the U.S. Energy Information Administration's
Residential Energy Consumption Survey (RECS). The dataset contains 5,686 U.S. households and captures
electricity usage, home characteristics, household demographics, and climate zone information.
Purpose: Understand what factors drive residential electricity consumption and cost, and build a predictive model
for household energy use.

**Questions I want to investigate:**

*   How does electricity consumption vary across climate zones?
*   Does household size or square footage predict energy use?
*   What role does income play in energy consumption?
*   Can we predict KWH usage from home and household characteristics?





# **Section 2: Load and Inspect the Data**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
# Load dataset
df = pd.read_csv('home_data_final.csv')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()
# Column names and data types
print(df.dtypes)
# Missing value check
print('Missing values per column:')
print(df.isnull().sum())

**Observation**: No missing values across any column. The dataset is clean and ready for analysis.
CLIMATE_REGION_PUB is the only string column; all others are numeric.

# **Section 3: Data Cleaning and Preparation**

In [ ]:
# Drop the household ID -- it carries no analytical value
df = df.drop(columns=['DOEID'])
# Map MONEYPY integer codes to readable income labels
income_labels = {
1: '&lt; $2,500',
2: '$2,500-$4,999',
3: '$5,000-$7,499',
4: '$7,500-$9,999',
5: '$10,000-$14,999',
6: '$15,000-$19,999',
7: '$20,000-$24,999',
8: '$25,000-$29,999'
}
df['INCOME_LABEL'] = df['MONEYPY'].map(income_labels)
# Map ENERGYASST to readable labels
df['ENERGYASST_LABEL'] = df['ENERGYASST'].map({0: 'No Assistance', 1: 'Receives Assistance'})
# Set climate zone as a proper ordered category for consistent plotting
climate_order = ['Cold/Very Cold', 'Mixed-Humid', 'Hot-Humid', 'Hot-Dry/Mixed-Dry', 'Marine']
df['CLIMATE_REGION_PUB'] = pd.Categorical(
df['CLIMATE_REGION_PUB'], categories=climate_order, ordered=True
)
print('Cleaned dataframe shape:', df.shape)
df.head(3)

**Cleaning steps taken:**

• Dropped DOEID (household ID) no analytical value.

• Decoded MONEYPY integer codes into human-readable income bracket labels.

• Decoded ENERGYASST binary flag into labeled categories.

• Converted CLIMATE_REGION_PUB to an ordered categorical type for consistent plot ordering.

# **Section 4: Descriptive Statistics**

In [ ]:
# Numerical summary statistics for key columns
num_cols = ['TOTSQFT_EN', 'KWH', 'DOLLAREL', 'TEMPHOME', 'TEMPGONE',
'TEMPNITE', 'NHSLDMEM', 'EL_TOTALS', 'BTUEL_TOTALS']
df[num_cols].describe().round(2)
# Categorical summary: Climate zone distribution
print('Climate Zone Distribution:')
print(df['CLIMATE_REGION_PUB'].value_counts().sort_index())
print()
print('Energy Assistance Participation:')
print(df['ENERGYASST_LABEL'].value_counts())

**Key observations:**

• Mean electricity consumption is about 10,751 KWH/year with a wide standard deviation (~7,600), indicating
substantial variation.

• The average annual electricity bill is approximately $1,385.

• Most households (2,008) are in Cold/Very Cold climate zones; Marine has the fewest (424).

• About 22% of households receive some form of energy assistance.

# **Section 5: Numerical Data Visualizations**

In [ ]:
def plot_hist_box(series, title, xlabel, color='steelblue'):
"""Reusable helper: side-by-side histogram and box plot for a numeric series."""
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(series.dropna(), bins=40, color=color, edgecolor='white', linewidth=0.5)
ax1.set_title(f'Distribution of {title}')
ax1.set_xlabel(xlabel)
ax1.set_ylabel('Count')
ax2.boxplot(series.dropna(), vert=False, patch_artist=True,
boxprops=dict(facecolor=color, alpha=0.7))
ax2.set_title(f'Box Plot: {title}')
ax2.set_xlabel(xlabel)
ax2.set_yticks([])
plt.tight_layout()
plt.show()
# Apply the helper to key numeric columns
plot_configs = [
(df['KWH'], 'Annual Electricity Consumption', 'KWH', 'steelblue'),
(df['DOLLAREL'], 'Annual Electricity Cost', 'USD ($)', 'darkorange'),
(df['TOTSQFT_EN'], 'Total Home Square Footage', 'Square Feet', 'mediumseagreen'),
]
for series, title, xlabel, color in plot_configs:
plot_hist_box(series, title, xlabel, color)


**Observations:**

• KWH is right-skewed: most households use modest electricity, but a long tail of high-consumption homes
pulls the mean upward.

• DOLLAREL mirrors the KWH distribution closely — cost tracks consumption.

• TOTSQFT_EN is also right-skewed; most homes are under 3,000 sq ft.

• Box plots confirm substantial outliers in all three variables, realistic for a national cross-section.

# **Section 6: Categorical Data Analysis**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
# Bar chart: household count by climate zone
climate_counts = df['CLIMATE_REGION_PUB'].value_counts().sort_index()
ax1.bar(climate_counts.index, climate_counts.values,
color='steelblue', edgecolor='white')
ax1.set_title('Households by Climate Zone')
ax1.set_xlabel('Climate Zone')
ax1.set_ylabel('Number of Households')
ax1.tick_params(axis='x', rotation=20)

# Pie chart: energy assistance participation
asst_counts = df['ENERGYASST_LABEL'].value_counts()
ax2.pie(asst_counts.values, labels=asst_counts.index, autopct='%1.1f%%',
colors=['steelblue', 'darkorange'], startangle=90)
ax2.set_title('Energy Assistance Program Participation')
plt.tight_layout()
plt.show()

**Observations:**

• Cold/Very Cold is the most represented climate zone (35% of households).

• Marine climate (Pacific Coast) is the least represented, consistent with its smaller geographic footprint.

• Roughly 22% of households participate in an energy assistance program.

# **Section 7: Grouped Analysis**

In [ ]:
# Group by climate zone and compute descriptive stats
grouped_climate = df.groupby('CLIMATE_REGION_PUB')[['KWH', 'DOLLAREL', 'TOTSQFT_EN']] \
.agg(['mean', 'median', 'std']).round(1)
grouped_climate
# Grouped bar chart: mean KWH and mean DOLLAREL by climate zone
climate_means = df.groupby('CLIMATE_REGION_PUB')[['KWH', 'DOLLAREL']].mean().sort_index()
x = np.arange(len(climate_means))
width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width/2, climate_means['KWH'], width, label='Mean KWH', color='steelblue', edgecolor='white')
ax.bar(x + width/2, climate_means['DOLLAREL'], width, label='Mean Cost ($)', color='darkorange', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(climate_means.index, rotation=15, ha='right')
ax.set_title('Mean Annual Electricity Consumption and Cost by Climate Zone')
ax.set_ylabel('Value (KWH or USD)')
ax.set_xlabel('Climate Zone')
ax.legend()
plt.tight_layout()
plt.show()
# Box plots: KWH distribution by climate zone
climate_groups = [df[df['CLIMATE_REGION_PUB'] == z]['KWH'].dropna() for z in climate_order]
fig, ax = plt.subplots(figsize=(11, 5))
bp = ax.boxplot(climate_groups, labels=climate_order, patch_artist=True, vert=True)
colors_list = ['steelblue', 'mediumseagreen', 'darkorange', 'tomato', 'plum']
for patch, color in zip(bp['boxes'], colors_list):
patch.set_facecolor(color)
patch.set_alpha(0.7)
ax.set_title('KWH Distribution by Climate Zone')
ax.set_xlabel('Climate Zone')
ax.set_ylabel('Annual Electricity Consumption (KWH)')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

**Observations**:

• Hot-Humid and Hot-Dry/Mixed-Dry zones have the highest mean KWH, driven by heavy air conditioning
demand.

• Marine zone households consume the least electricity mild year-round temperatures reduce heating and
cooling demand.

• Cold/Very Cold zones show high variance, likely because electric vs. gas heating choices differ widely.

• Cost closely tracks consumption across all zones.

# **Section 8: New Feature Engineering**

In [ ]:
# New feature: Energy Intensity -- KWH consumed per square foot
# This normalizes consumption by home size, making homes comparable regardless of size.
df['ENERGY_INTENSITY'] = df['KWH'] / df['TOTSQFT_EN']
print('Energy Intensity (KWH/sqft) -- Summary:')
print(df['ENERGY_INTENSITY'].describe().round(3))
# Compare energy intensity across climate zones
intensity_by_climate = df.groupby('CLIMATE_REGION_PUB')['ENERGY_INTENSITY'].mean().sort_index()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(intensity_by_climate.index, intensity_by_climate.values,
color=['steelblue', 'mediumseagreen', 'darkorange', 'tomato', 'plum'],
edgecolor='white')
ax.set_title('Mean Energy Intensity (KWH per Square Foot) by Climate Zone')
ax.set_xlabel('Climate Zone')
ax.set_ylabel('KWH per Square Foot')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

**Why this feature matters:** Raw KWH is confounded by home size. ENERGY_INTENSITY (KWH per square foot)
controls for this, revealing which climate zones are inherently more energy-demanding per unit of living space.

**Observation:** Even after normalizing for size, Hot-Humid and Hot-Dry/Mixed-Dry zones show the highest energy
intensity, confirming that climate (not just home size) is a major driver of electricity demand.

# **Section 9: Trends Across Income Levels**

In [ ]:
# Define income bracket order
income_order = list(income_labels.values())
df['INCOME_LABEL'] = pd.Categorical(df['INCOME_LABEL'], categories=income_order, ordered=True)
income_trends = df.groupby('INCOME_LABEL')[['KWH', 'DOLLAREL', 'TOTSQFT_EN']].mean().reset_index()
# Standardize (z-score) so three different-scale variables can be plotted on one axis
def standardize(series):
return (series - series.mean()) / series.std()

for col in ['KWH', 'DOLLAREL', 'TOTSQFT_EN']:
income_trends[col + '_z'] = standardize(income_trends[col])
fig, ax = plt.subplots(figsize=(12, 5))
for col, label, color in [('KWH_z', 'KWH (standardized)', 'steelblue'),
('DOLLAREL_z', 'Cost (standardized)', 'darkorange'),
('TOTSQFT_EN_z','Sq Ft (standardized)', 'mediumseagreen')]:
ax.plot(income_trends['INCOME_LABEL'], income_trends[col],
marker='o', label=label, color=color, linewidth=2)
ax.set_title('Trends in KWH, Cost, and Home Size Across Income Brackets (Standardized)')
ax.set_xlabel('Monthly Household Income Bracket')
ax.set_ylabel('Standardized Value (z-score)')
ax.tick_params(axis='x', rotation=30)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.legend()
plt.tight_layout()
plt.show()

**Observations:**

• All three variables: KWH, electricity cost, and square footage generally rise with income.

• The lowest income bracket shows below-average energy use, likely reflecting smaller homes and
conservative consumption.

• Square footage tracks income more tightly than KWH does, suggesting wealthier households own larger
homes that drive higher energy use.

# **Section 10: Variable Relationships**

In [ ]:
# Correlation matrix for key numeric variables
corr_cols = ['KWH', 'DOLLAREL', 'TOTSQFT_EN', 'NHSLDMEM', 'EL_TOTALS', 'MONEYPY',
'TEMPHOME', 'TEMPGONE', 'TEMPNITE', 'ENERGY_INTENSITY']
corr_matrix = df[corr_cols].corr().round(2)
# === Something New: Annotated correlation heatmap using matplotlib ===
# (Goes beyond class examples -- manually rendering a heatmap with cell annotations
# using imshow + text overlays, without seaborn)
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr_matrix.values, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Correlation Coefficient')
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)
# Annotate each cell with the correlation value
for i in range(len(corr_cols)):
for j in range(len(corr_cols)):
val = corr_matrix.iloc[i, j]
text_color = 'white' if abs(val) &gt; 0.6 else 'black'
ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=text_color)
ax.set_title('Correlation Heatmap -- Key Energy Variables', fontsize=13, pad=12)
plt.tight_layout()

plt.show()
# Scatter plots: KWH vs. key predictors
scatter_pairs = [
('TOTSQFT_EN', 'Total Square Footage', 'mediumseagreen'),
('EL_TOTALS', 'Number of Electricity Components', 'darkorange'),
('NHSLDMEM', 'Household Members', 'steelblue'),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (xcol, xlabel, color) in zip(axes, scatter_pairs):
ax.scatter(df[xcol], df['KWH'], alpha=0.15, s=10, color=color)
m, b = np.polyfit(df[xcol], df['KWH'], 1)
x_line = np.linspace(df[xcol].min(), df[xcol].max(), 100)
r_val = df[[xcol, 'KWH']].corr().iloc[0, 1]
ax.plot(x_line, m * x_line + b, color='black', linewidth=1.5,
linestyle='--', label=f'r = {r_val:.2f}')
ax.set_xlabel(xlabel)
ax.set_ylabel('Annual KWH')
ax.set_title(f'KWH vs. {xlabel}')
ax.legend(fontsize=9)
plt.suptitle('Scatter Plots: KWH vs. Key Predictors', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Observations:**

• DOLLAREL has the strongest correlation with KWH (r ≈ 0.88) cost is derived from consumption.

• EL_TOTALS correlates meaningfully with KWH (r ≈ 0.41): more appliances/systems means more electricity.

• TOTSQFT_EN shows a moderate positive relationship (r ≈ 0.37): larger homes consume more.

• NHSLDMEM has a weaker but positive correlation (r ≈ 0.29): more people adds to demand.

• Temperature settings show weak individual correlations.

# **Section 11: Hypothesis**
**Hypothesis:** Homes with more electricity-consuming components (EL_TOTALS) and larger square footage
(TOTSQFT_EN) will have significantly higher annual electricity consumption (KWH).

**Reasoning:** The number of appliances and systems in a home directly determines how many devices draw from
the electrical grid. Square footage determines how much space must be heated, cooled, and lit. Together, these
two variables should explain a meaningful portion of the variance in KWH consumption. Neither factor alone is
sufficient, a large home with minimal appliances may still conserve energy, while a small home packed with
equipment may consume heavily.

# **Section 12: Linear Regression**

In [ ]:
# Feature and outcome selection
features = ['TOTSQFT_EN', 'EL_TOTALS', 'NHSLDMEM']
outcome = 'KWH'
X = df[features]

y = df[outcome]
# Train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Fit linear regression model
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
# Model evaluation
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print('=== Linear Regression Results ===')
print(f'Features: {features}')
print(f'Outcome: {outcome}')
print()
for feat, coef in zip(features, model.coef_):
print(f' Coefficient [{feat}]: {coef:.4f}')
print(f' Intercept: {model.intercept_:.2f}')
print()
print(f' R^2 Score: {r2:.4f}')
print(f' RMSE: {rmse:.2f} KWH')
# Plot 1: Predicted vs. Actual values
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.scatter(y_test, y_pred, alpha=0.2, s=10, color='steelblue')
max_val = max(y_test.max(), y_pred.max())
ax1.plot([0, max_val], [0, max_val], color='red', linestyle='--',
linewidth=1.5, label='Perfect Fit')
ax1.set_xlabel('Actual KWH')
ax1.set_ylabel('Predicted KWH')
ax1.set_title(f'Predicted vs. Actual KWH (R^2 = {r2:.3f})')
ax1.legend()
# Plot 2: Regression line -- KWH vs. TOTSQFT_EN, holding others at mean
sqft_range = np.linspace(df['TOTSQFT_EN'].min(), df['TOTSQFT_EN'].max(), 200)
mean_el = df['EL_TOTALS'].mean()
mean_nhsld = df['NHSLDMEM'].mean()
X_line = pd.DataFrame({'TOTSQFT_EN': sqft_range,
'EL_TOTALS': mean_el,
'NHSLDMEM': mean_nhsld})
y_line = model.predict(X_line)
ax2.scatter(df['TOTSQFT_EN'], df['KWH'], alpha=0.1, s=8,
color='steelblue', label='Observed')
ax2.plot(sqft_range, y_line, color='red', linewidth=2,
label='Regression Line (EL_TOTALS, NHSLDMEM at mean)')
ax2.set_xlabel('Total Square Footage')
ax2.set_ylabel('Annual Electricity (KWH)')
ax2.set_title('KWH vs. Square Footage -- Regression Line')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

# **Section 13: Regression Discussion**

**What the regression results suggest**

The model uses square footage (TOTSQFT_EN), number of electricity components (EL_TOTALS), and
household size (NHSLDMEM) to predict annual electricity consumption. Each additional square foot adds a small
but measurable increment to KWH. Each additional electricity-using component contributes a larger jump —
appliances draw directly from the grid. Household size adds a smaller incremental effect.

**Does the model do a good job?**

The R2 score indicates the model explains roughly 28-32% of the variance in KWH. That is meaningful but
modest. The predicted vs. actual scatter plot shows reasonable alignment at low consumption levels, but the
model struggles with high-consumption outliers above ~20,000 KWH.

**Limitations**


• Missing variables: Electric vs. gas heating source is a major omitted driver. A fully electric home has far
higher KWH than an identical gas-heated home.

• Nonlinearity: The KWH-to-sqft relationship is not strictly linear; returns likely increase at larger home sizes.

• Climate zone not included: Regional climate is a strong confounder that could significantly improve the
model.

• Multicollinearity: TOTSQFT_EN and EL_TOTALS are correlated, which can make individual coefficient
estimates less stable.

**Possible improvements**

• Add climate zone as a one-hot encoded feature.

• Include heating fuel type.

• Explore polynomial or log transformations of skewed predictors.

• Use a regularized model (Ridge/Lasso) or tree-based model to capture nonlinear effects.

# **Section 14: Reflection and Conclusion**
**Summary of findings**

• Climate zone is a strong categorical predictor of energy intensity. Hot-Humid and Hot-Dry zones consume
the most electricity per square foot, driven by cooling demand.

• Larger homes and homes with more electricity-drawing components consume more energy, but raw square
footage alone is an incomplete predictor.

• Income correlates positively with energy use, mostly through the channel of larger home size.

• ENERGY_INTENSITY (KWH/sqft) was the most analytically useful engineered feature, enabling fair
comparisons across homes of different sizes.
What I learned

• Cross-sectional datasets require creative adaptation of 'trends over time' analyses, ordered categorical
variables can serve a similar structural purpose.

• Building a manually annotated correlation heatmap in matplotlib (without seaborn) was a useful exercise in
low-level visualization control.


• A simple three-variable OLS regression explains roughly 30% of variance in KWH, underscoring how much
explanatory power lies in variables not fully captured here (e.g., heating fuel type, equipment efficiency).


**Future directions**

• Merge with utility rate data to explore price elasticity of electricity demand by region.

• Build a multi-feature model including climate zone dummy variables and heating fuel type.

• Explore clustering to identify household archetypes and compare their energy profiles.